<a href="https://colab.research.google.com/github/surajjeoor/masai_assignments/blob/main/production_LLM_apps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q google-generativeai streamlit pyngrok

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-resource-manager 1.19.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-language 2.22.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-monitoring 2.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
g

In [14]:
#cell imports
import os
import json
import datetime
import time
import google.generativeai as palm
import textwrap
from getpass import getpass
print("All import successful")

All import successful


In [6]:
GEMINI_KEY=getpass("Enter your Gemini API Key")
os.environ["GEMINI_API_KEY"]=GEMINI_KEY
api_model="gemini-3.1-flash-lite"
model= palm.GenerativeModel(api_model)
print(f"Model {api_model} loaded successfully")

Enter your Gemini API Key··········
Model gemini-3.1-flash-lite loaded successfully


In [10]:
import google.generativeai as palm
import os

# Configure the API key using the value stored in your environment
palm.configure(api_key=os.environ.get("GEMINI_API_KEY"))

#generate first api call
response=model.generate_content("You are a shopease customer support advisor"
                                    "A customer asks: what is the return policy?"
                                    "Please reply in 2 sentences")
#----------------reply of the model
print("="*60)
print("RESPONSE OF THE MODEL")
print("="*60)
print(response.text)
print("\n")

#--------------response metadata
print("\n")
print("="*60)
print("RESPONSE METADATA")
print("="*60)
print(f"Model used: {api_model}")
print(f"finish reason {response.candidates[0].finish_reason.name}")
print(f"tokens usage:{response.usage_metadata}")

RESPONSE OF THE MODEL
You can return any item within 30 days of purchase, provided it remains in its original packaging and unused condition. Please visit our online returns portal to initiate the process and receive your prepaid shipping label.




RESPONSE METADATA
Model used: gemini-3.1-flash-lite
finish reason STOP
tokens usage:prompt_token_count: 26
candidates_token_count: 41
total_token_count: 67



In [15]:
#we are printing the response from the model when its not streaming
print("Non streaming reply: full text appear at once")
print("="*60)
start=time.time()
response_wo_stream=model.generate_content("List 5 tips for being more productive. Be concise")
elapsed=time.time()-start

print(response_wo_stream.text)
print(f"Waited {elapsed:.2f} seconds to get the response")


Non streaming reply: full text appear at once
1. **Prioritize tasks:** Use the Eisenhower Matrix to focus on what is urgent and important.
2. **Time block:** Dedicate specific chunks of time to individual tasks to avoid multitasking.
3. **Use the Pomodoro Technique:** Work in 25-minute intervals followed by short breaks to maintain focus.
4. **Minimize distractions:** Silence notifications and clear your workspace to stay in a flow state.
5. **Eat the frog:** Complete your most difficult or important task first thing in the morning.
Waited 25.69 seconds to get the response


In [16]:
print(" streaming reply: chunks of responses appear as they get generated from the model")
print("="*60)
start=time.time()
response_stream=model.generate_content("List 5 tips for being more productive. Be concise",
                                       stream=True)
first_chunk_time=None

for chunk in response_stream:
    if chunk.text:
        if chunk is not None:
            first_chunk_time=time.time()-start
        print(chunk, end="", flush=True)
print(f"time took for first chunk: {first_chunk_time} which is fast")

 streaming reply: chunks of responses appear as they get generated from the model
response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "text": "1. **Prioritize tasks:** Use the Eisenhower Matrix to focus on what"
              }
            ],
            "role": "model"
          },
          "index": 0
        }
      ],
      "usage_metadata": {
        "prompt_token_count": 12,
        "candidates_token_count": 15,
        "total_token_count": 27
      },
      "model_version": "gemini-3.1-flash-lite"
    }),
)response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "text": " is urgent and important.\n2. **Time-block your day:** Dedicate specific chunk

In [17]:
# Cell: streaming
# With streaming — text appears token by token as the model generates it

print("Streaming reply (tokens arrive live):")
print("-" * 60)

start = time.time()
stream = model.generate_content(
    "List 5 tips for writing a good product return request. Be concise.",
    stream=True
)

# Each chunk contains a piece of the response text
first_chunk_time = None
for chunk in stream:
    if chunk.text:
        if first_chunk_time is None:
            first_chunk_time = time.time() - start
        print(chunk.text, end="", flush=True)

print(f"\n\n[First token arrived in {first_chunk_time:.1f}s — much faster!]")

Streaming reply (tokens arrive live):
------------------------------------------------------------
Here are 5 tips for writing an effective product return request:

1.  **Include Essential Details:** Clearly state your order number, the date of purchase, and the specific item(s) you wish to return.
2.  **State the Reason Clearly:** Briefly explain why you are returning the item (e.g., damaged, incorrect size, or defective) to help the support team process your request faster.
3.  **Provide Photographic Evidence:** If the item arrived damaged or defective, attach clear photos to your email to expedite your claim.
4.  **Reference the Policy:** Briefly mention that you have reviewed the company’s return policy to show that your request meets their criteria.
5.  **Be Professional and Concise:** Keep your tone polite and get straight to the point to ensure your request is easy for support staff to read and action.

[First token arrived in 36.8s — much faster!]


In [18]:
SYSTEM_PROMPT = (
    "You are the ShopEase Support Agent. "
    "Help customers with orders, returns, and account queries. "
    "Keep replies brief and friendly."
)

# start_chat() creates a session with optional initial history
chat_session = model.start_chat(history=[])

def chat(user_conv:str)->str:
  full_msg=f"[SYSTEM: {SYSTEM_PROMPT}]\n\nCustomer: {user_conv}"
  response_chat=chat_session.send_message(full_msg)
  return response_chat.text

# Simulate a conversation — the model should remember context
conversations = [
    "Hi, I'm Priya. I ordered a phone case last week.",
    "The case arrived damaged. I want to return it.",
    "What is my name again?",   # tests memory
]

for con in conversations:
  ("="*60)
  print(f"Customer: {con}")
  print(f"ShopEaseBot: {chat(con)}")
  (print("="*60))

Customer: Hi, I'm Priya. I ordered a phone case last week.
ShopEaseBot: Hi Priya! I'd be happy to help you with that. Could you please provide your order number?
Customer: The case arrived damaged. I want to return it.
ShopEaseBot: I'm so sorry to hear that, Priya. I can certainly help you with a return for your damaged item. 

Could you please provide your order number so I can pull up your account details and start the process for you?
Customer: What is my name again?
ShopEaseBot: Hi Priya! Your name is Priya. Is there anything else I can help you with regarding your order?


In [20]:
import time
from google.api_core.exceptions import ResourceExhausted, PermissionDenied, DeadlineExceeded

def safe_generate(prompt: str, retries: int = 3, timeout: float = 30.0) -> str:
  for attempt in range(1, retries + 1):
    try:
      response_ex_handling = model.generate_content(
          prompt,
          generation_config={"temperature": 0.25},
          request_options={'timeout': timeout}
      )
      return response_ex_handling.text

    except ResourceExhausted:
      # 429 - rate limit
      wait = 2 ** (attempt - 1)
      print(f"[Attempt {attempt}] rate limit hit. Retrying in {wait} seconds")
      time.sleep(wait)

    except PermissionDenied:
      # 403 - access denied
      print("Permission to access model has been denied. Please check your credentials and retry again.")
      return "Permission denied."

    except DeadlineExceeded:
      wait = 2 ** (attempt - 1)
      print(f"[Attempt {attempt}] request timed out. Retrying in {wait} seconds")
      time.sleep(wait)

    except Exception as e:
      print(f"[Attempt {attempt}] unexpected error: {type(e).__name__}: {e}")
      if attempt == retries:
        return "Something went wrong. Please retry after some time."
      time.sleep(1)

  return "Service is unavailable. Please try again."

# Test with a real call
print("Testing safe_generate():")
reply = safe_generate("What is ShopEase's return window? Reply in one sentence.")
print(f"Reply: {reply}")

Testing safe_generate():
[Attempt 1] request timed out. Retrying in 1 seconds
[Attempt 2] request timed out. Retrying in 2 seconds
Reply: ShopEase offers a 30-day return window for all eligible items.


In [25]:
# Cell: write-app
# Write the Streamlit app to a .py file
# Note: the API key is passed via environment variable — not hardcoded

app_code = '''
import os
import time
import json
import datetime
import google.generativeai as genai
import streamlit as st

# ── Configuration ──────────────────────────────────────────────────────────
API_KEY    = os.environ.get("GEMINI_API_KEY", "")
MODEL_NAME = "gemini-3.1-flash-lite"
LOG_FILE   = "shopease_conversations.jsonl"

genai.configure(api_key=API_KEY)
gemini_model = genai.GenerativeModel(MODEL_NAME)

SYSTEM_PROMPT = (
    "You are the ShopEase Support Agent. "
    "Help customers with orders, returns, and account queries. "
    "Keep replies clear, friendly, and under 3 sentences."
)

# ── Logging ────────────────────────────────────────────────────────────────
def log_turn(user_msg, assistant_msg, latency_ms):
    entry = {
        "timestamp":  datetime.datetime.utcnow().isoformat() + "Z",
        "model":      MODEL_NAME,
        "user":       user_msg,
        "assistant":  assistant_msg,
        "latency_ms": round(latency_ms),
    }
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(entry) + "\\n")

# ── Streamlit UI ───────────────────────────────────────────────────────────
st.set_page_config(page_title="ShopEase Support", page_icon="🛍️")
st.title("🛍️ ShopEase Support Chat")
st.caption(f"Powered by {MODEL_NAME}")

# Initialise session state — persists between Streamlit re-runs
if "messages" not in st.session_state:
    st.session_state.messages = []   # list of {role, content}

if "chat_session" not in st.session_state:
    st.session_state.chat_session = gemini_model.start_chat(history=[])

# Display conversation history
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# Accept new user input
if user_input := st.chat_input("Ask ShopEase support..."):

    # Show user message immediately
    st.session_state.messages.append({"role": "user", "content": user_input})
    with st.chat_message("user"):
        st.write(user_input)

    # Stream assistant reply
    with st.chat_message("assistant"):
        t0 = time.time()
        full_message = f"[SYSTEM: {SYSTEM_PROMPT}]\\n\\nCustomer: {user_input}"

        # Use streaming — write_stream handles the generator automatically
        def token_stream():
            stream = st.session_state.chat_session.send_message(
                full_message, stream=True
            )
            for chunk in stream:
                if chunk.text:
                    yield chunk.text

        reply_text = st.write_stream(token_stream())
        latency_ms = (time.time() - t0) * 1000

    # Save to history and log
    st.session_state.messages.append({"role": "assistant", "content": reply_text})
    log_turn(user_input, reply_text, latency_ms)
'''

with open("shopease_chat.py", "w") as f:
    f.write(app_code.strip())

print("shopease_chat.py written. Ready to launch.")

shopease_chat.py written. Ready to launch.


In [27]:
SYSTEM_PROMPT = "You are a friendly support agent for ShopEase. Answer only from the facts given."

history = [
    {"role": "user", "content": "Hi, I am a returning customer."},
    {"role": "assistant", "content": "Welcome back! How can I help?"},
    {"role": "user", "content": "My order is 041."},
    {"role": "assistant", "content": "Thanks. What happened to order 041?"},
    {"role": "user", "content": "It arrived damaged."},
    {"role": "assistant", "content": "Sorry to hear that. When was it delivered?"},
    {"role": "user", "content": "Four days ago."},
    {"role": "assistant", "content": "You can return it within 10 days if unused."},
    {"role": "user", "content": "It is unused."},
    {"role": "assistant", "content": "Great. Keep it in the original packing."},
    {"role": "user", "content": "How long does the refund take?"},
    {"role": "assistant", "content": "Refunds take 5 to 7 business days."},
]


def build_messages(system_prompt, history, user_message, k=10):
    # TODO: append the new user message to history
    # TODO: return the system message followed by the last k history entries
    messages = []
    messages.append({"role": "system", "content": system_prompt})
    for message in history[-k:]:
        messages.append(message)
    messages.append({"role": "user", "content": user_message})
    return messages


if __name__ == "__main__":
    messages = build_messages(SYSTEM_PROMPT, history, "Can I return it?", k=10)
    if not messages:
        print("build_messages returned nothing yet")
    else:
        print("Messages sent:", len(messages))
        print("First role:", messages[0]["role"])
        print("Oldest kept:", messages[1]["role"], "|", messages[1]["content"])
        print("Newest kept:", messages[-1]["role"], "|", messages[-1]["content"])
        print("History length:", len(history))

Messages sent: 12
First role: system
Oldest kept: user | My order is 041.
Newest kept: user | Can I return it?
History length: 12


In [47]:
import json
import statistics
import re

LOG_TEXT = """
{
  "model": "gpt-5.4-mini",
  "question": "How long do refunds take?",
  "prompt_tokens": 40,
  "completion_tokens": 60,
  "latency": 1.2,
  "error": null
}
{
  "model": "gpt-5-mini",
  "question": "Is delivery free?",
  "prompt_tokens": 30,
  "completion_tokens": 50,
  "latency": 0.9,
  "error": null
}
{
  "model": "gpt-5.4-mini",
  "question": "Do you price match?",
  "prompt_tokens": 50,
  "completion_tokens": 40,
  "latency": 1.5,
  "error": null
}
{
  "model": "gpt-5.4-mini",
  "question": "Where is my order?",
  "prompt_tokens": 0,
  "completion_tokens": 0,
  "latency": 0.2,
  "error": "AuthenticationError"
}
{
  "model": "gpt-5-mini",
  "question": "Can I return shoes?",
  "prompt_tokens": 60,
  "completion_tokens": 70,
  "latency": 1.1,
  "error": null
}
{
  "model": "gpt-5-mini",
  "question": "Hello",
  "prompt_tokens": 0,
  "completion_tokens": 0,
  "latency": 0.4,
  "error": "AuthenticationError"
}
""".strip()

# (input price, output price) per 1 million tokens, in US dollars
PRICES = {"gpt-5.4-mini": (0.75, 4.50), "gpt-5-mini": (0.25, 2.00)}


def summarise_log(log_text, prices):
    summary = {}

    # Use regular expressions to extract complete JSON objects enclosed by curly braces
    json_blocks = re.findall(r'\{.*?\}', log_text, re.DOTALL)

    for block in json_blocks:
        record = json.loads(block)
        model = record["model"]

        if model not in summary:
            summary[model] = {
                "requests": 0,
                "errors": 0,
                "latencies": [],
                "cost": 0.0
            }

        summary[model]["requests"] += 1
        if record.get("error") is not None:
            summary[model]["errors"] += 1

        summary[model]["latencies"].append(record["latency"])

        input_price, output_price = prices.get(model, (0.0, 0.0))
        cost = (
            (record["prompt_tokens"] * input_price) +
            (record["completion_tokens"] * output_price)
        ) / 1000000.0
        summary[model]["cost"] += cost

    for model in summary:
        latencies = summary[model]["latencies"]
        summary[model]["avg_latency"] = statistics.mean(latencies) if latencies else 0.0
        del summary[model]["latencies"]

    return summary


if __name__ == "__main__":
    summary = summarise_log(LOG_TEXT, PRICES)
    if not summary:
        print("summarise_log returned nothing yet")
    for model in sorted(summary):
        s = summary[model]
        print(f"{model}: requests={s['requests']} errors={s['errors']} "
              f"avg_latency={s['avg_latency']:.2f}s cost=${s['cost']:.7f}")

gpt-5-mini: requests=3 errors=1 avg_latency=0.80s cost=$0.0002625
gpt-5.4-mini: requests=3 errors=1 avg_latency=0.97s cost=$0.0005175


In [48]:
CHUNKS = [
    (0.42, None),
    (0.81, "Shop"),
    (0.83, "Ease"),
    (0.90, " accepts"),
    (1.05, " returns"),
    (1.20, " within"),
    (1.32, " 10"),
    (1.40, " days."),
    (1.55, None),
]


def consume_stream(chunks):
    result = {"reply": "", "chunks": 0, "ttft": None, "total": None}
    # TODO: loop over chunks, skip empty content, record the TTFT,
    # join the content pieces and record the total time
    for chunk in chunks:
        if chunk[1] is not None:
            result["reply"] += chunk[1]
            result["chunks"] += 1
            result["ttft"] = chunk[0]
        result["total"] = chunk[0]
    return result


if __name__ == "__main__":
    result = consume_stream(CHUNKS)
    if result["ttft"] is None:
        print("consume_stream is not finished yet")
    else:
        print("Reply:", result["reply"])
        print("Content chunks:", result["chunks"])
        print(f"TTFT: {result['ttft']:.2f}s")
        print(f"Total: {result['total']:.2f}s")

Reply: ShopEase accepts returns within 10 days.
Content chunks: 7
TTFT: 1.40s
Total: 1.55s
